## Проверка data-pipeline


In [7]:
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

from src.dataloaders import create_dataloaders, create_test_dataloader


## Preprocessing с recommended heuristics

Проверяем новый режим: к train добавляются только `cabinet` и `dressing_room`, а количество примеров добирается до среднего размера класса.


In [13]:
# Проверяем, что heuristics добрали нужные классы до среднего размера
train_df = pd.read_csv("../data/processed/train_df.csv")


In [14]:
# Создаём DataLoader на основе processed CSV и локальных изображений
train_loader, val_loader = create_dataloaders(
    batch_size=32,
    num_workers=0,
    image_size=224,
    use_weighted_sampling=False
)

test_loader = create_test_dataloader(
    batch_size=32,
    num_workers=0,
    image_size=224
)

print("train объектов:", len(train_loader.dataset))
print("val объектов:", len(val_loader.dataset))
print("test объектов для предсказания:", len(test_loader.dataset))


train объектов: 4632
val объектов: 477
test объектов для предсказания: 47791


In [ ]:
train_balance_df = check_loader_class_balance(train_loader)
train_balance_df

## Confusion matrix для класса `предметы интерьера / быт.техника`

Проверяем, с какими классами модель путает класс 17 на validation set.


In [17]:
import json

from models.resnet18.resnet18 import build_resnet18
from src.labels import load_label_mapping

target_class_id = 17
metrics_path = PROJECT_ROOT / "reports" / "metrics" / "resnet18" / "resnet18_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
checkpoint_path = Path(metrics["checkpoint"])

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

eval_model = build_resnet18(
    num_classes=int(checkpoint.get("num_classes", 19)),
    pretrained=False,
).to(device)
eval_model.load_state_dict(checkpoint["model_state_dict"])
eval_model.eval()

y_true = []
y_pred = []
with torch.inference_mode():
    for batch_images, batch_targets in val_loader:
        batch_images = batch_images.to(device)
        outputs = eval_model(batch_images)
        predictions = outputs.argmax(dim=1).cpu().tolist()
        y_pred.extend(predictions)
        y_true.extend(batch_targets.tolist())

label_mapping = load_label_mapping()
target_label = label_mapping[target_class_id]
class_ids = list(range(int(checkpoint.get("num_classes", 19))))

confusion_df = (
    pd.crosstab(
        pd.Series(y_true, name="true_class"),
        pd.Series(y_pred, name="predicted_class"),
    )
    .reindex(index=class_ids, columns=class_ids, fill_value=0)
)

tp = int(confusion_df.loc[target_class_id, target_class_id])
fn = int(confusion_df.loc[target_class_id].sum() - tp)
fp = int(confusion_df[target_class_id].sum() - tp)
tn = int(confusion_df.to_numpy().sum() - tp - fn - fp)

binary_confusion_df = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=[f"true {target_class_id}: {target_label}", "true other classes"],
    columns=[f"predicted {target_class_id}: {target_label}", "predicted other classes"],
)

target_false_negatives = (
    confusion_df.loc[target_class_id]
    .drop(index=target_class_id)
    .rename("count")
    .reset_index()
    .rename(columns={"predicted_class": "predicted_class_id"})
)
target_false_negatives["predicted_label"] = target_false_negatives["predicted_class_id"].map(label_mapping)
target_false_negatives = target_false_negatives[target_false_negatives["count"] > 0].sort_values("count", ascending=False)

target_false_positives = (
    confusion_df[target_class_id]
    .drop(index=target_class_id)
    .rename("count")
    .reset_index()
    .rename(columns={"true_class": "true_class_id"})
)
target_false_positives["true_label"] = target_false_positives["true_class_id"].map(label_mapping)
target_false_positives = target_false_positives[target_false_positives["count"] > 0].sort_values("count", ascending=False)

class17_metrics = next(
    row for row in metrics["best_epoch_metrics"]["per_class_metrics"]
    if row["class_id"] == target_class_id
)

print("checkpoint:", checkpoint_path)
print("target class:", target_class_id, target_label)
print("f1:", round(class17_metrics["f1"], 4))
print("accuracy:", round(class17_metrics["accuracy"], 4))
print("support:", class17_metrics["support"])

display(binary_confusion_df)
display(target_false_negatives)
display(target_false_positives)


checkpoint: /Users/user/projects/Python/room_type_classifier/outputs/models/resnet18/resnet18_2026-05-06_16-24-09_best.pt
target class: 17 предметы интерьера / быт.техника
f1: 0.4103
accuracy: 0.3478
support: 23


,predicted 17: предметы интерьера / быт.техника,predicted other classes
true 17: предметы интерьера / быт.техника,8,15
true other classes,8,446


,predicted_class_id,count,predicted_label
11,11,7,гардеробная / кладовая / постирочная
3,3,2,гостиная
10,10,2,коридор / прихожая
0,0,1,кухня / столовая
4,4,1,спальня
15,15,1,подъезд / лестничная площадка
17,18,1,комната без мебели


,true_class_id,count,true_label
7,7,2,ванная комната
11,11,2,гардеробная / кладовая / постирочная
1,1,1,кухня-гостиная
3,3,1,гостиная
4,4,1,спальня
16,16,1,другое
